# OpenAI Function Calling


In [41]:
from IPython.display import display, HTML
display(HTML(
"""
<a target="_blank" href="https://colab.research.google.com/github/pedrodiamel/agents-mini-course/blob/course/books/aula_04_openai_functions.ipynb">
  <img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/>
</a>
"""
))

In [1]:
import os
import openai

from dotenv import load_dotenv, find_dotenv
_ = load_dotenv(find_dotenv()) # read local .env file
openai.api_key = os.environ['OPENAI_API_KEY']

In [2]:
import requests

city = "Recife"
unit = "celsius"

# Get APIKey https://home.openweathermap.org/
WEATHER_API_KEY = os.environ['WEATHER_API_KEY']
response = requests.get(f"https://api.openweathermap.org/data/2.5/weather?q={city}&appid={WEATHER_API_KEY}&units={unit}")
data = response.json()
data


{'coord': {'lon': -34.8811, 'lat': -8.0539},
 'weather': [{'id': 500,
   'main': 'Rain',
   'description': 'light rain',
   'icon': '10n'}],
 'base': 'stations',
 'main': {'temp': 298.17,
  'feels_like': 299.03,
  'temp_min': 298.17,
  'temp_max': 298.17,
  'pressure': 1017,
  'humidity': 88,
  'sea_level': 1017,
  'grnd_level': 1015},
 'visibility': 10000,
 'wind': {'speed': 1.54, 'deg': 220},
 'rain': {'1h': 0.21},
 'clouds': {'all': 20},
 'dt': 1787012401,
 'sys': {'type': 1,
  'id': 8426,
  'country': 'BR',
  'sunrise': 1786955283,
  'sunset': 1786997969},
 'timezone': -10800,
 'id': 3390760,
 'name': 'Recife',
 'cod': 200}

In [3]:
import json

# Example dummy function hard coded to return the same weather
# In production, this could be your backend API or an external API
def get_current_weather(location, unit="fahrenheit"):
    """Get the current weather in a given location"""

    # Chama API

    # # Dummy response
    # weather_info = {
    #     "location": location,
    #     "temperature": "72",
    #     "unit": unit,
    #     "forecast": ["sunny", "windy"],
    # }

    units = {"fahrenheit": "imperial", "celsius": "metric", "kelvin": "standard"}
    if unit not in units:
        raise ValueError("Invalid unit. Must be 'fahrenheit', 'celsius', or 'kelvin'.")

    response = requests.get(f"https://api.openweathermap.org/data/2.5/weather?q={location}&appid={WEATHER_API_KEY}&units={units[unit]}")
    data = response.json()
    
    weather_info = {
        "location": location,
        "temperature": data['main']['temp'],
        "unit": unit,
        "forecast": [data['weather'][0]['description']],
    }

    return weather_info #json.dumps(weather_info)

In [4]:
get_current_weather("Recife", "celsius")

{'location': 'Recife',
 'temperature': 25.02,
 'unit': 'celsius',
 'forecast': ['light rain']}

In [5]:
# define a function
# https://platform.openai.com/docs/guides/function-calling

# tools = [
#     {
#         "type": "function",
#         "function": {
#             "name": "get_current_weather",
#             "description": "Get the current weather in a given location",
#             "parameters": {
#                 "type": "object",
#                 "properties": {
#                     "location": {
#                         "type": "string",
#                         "description": "The city and state, e.g. San Francisco, CA",
#                     },
#                     "unit": {"type": "string", "enum": ["celsius", "fahrenheit"]},
#                 },
#                 "required": ["location"],
#             },
#         }
#     }
# ]

tools = [
    {
        "type": "function",
        "name": "get_current_weather",
        "description": "Obtém o clima atual para uma localização.",
        "parameters": {
            "type": "object",
            "properties": {
                "location": {
                    "type": "string",
                    "description": "Cidade ou coordenadas, ex: Recife ou -8.05,-34.88"
                }
            },
            "required": ["location"],
            "additionalProperties": False
        }
    }
]

In [6]:
mapping_tool_function = {
    "get_current_weather": get_current_weather
}

def execute_tool(tool_name, tool_args):
    
    result = mapping_tool_function[tool_name](**tool_args)

    if result is None:
        result = "The operation completed but didn't return any results."
        
    elif isinstance(result, list):
        result = ', '.join(result)
        
    elif isinstance(result, dict):
        # Convert dictionaries to formatted JSON strings
        result = json.dumps(result) #, indent=2
    
    else:
        # For any other type, convert using str()
        result = str(result)
    
    return result


In [7]:
import openai

llm_model = "gpt-4o-mini"
client = openai.OpenAI()


In [10]:

from prompts import SYSTEM_PROMPT
print(SYSTEM_PROMPT)

def process_query(query: str) -> str:
    """
    Envia uma pergunta para o modelo e executa tools
    enquanto o modelo solicitar chamadas de função.
    """

    response = client.responses.create(
        model=llm_model,
        instructions=SYSTEM_PROMPT,
        input=query,
        tools=tools,
    )

    while True:
        tool_outputs = []

        # response.output pode conter:
        # - mensagens de texto
        # - function calls
        # - outros tipos de output
        for item in response.output:

            if item.type == "function_call":
                tool_name = item.name

                try:
                    tool_args = json.loads(item.arguments)
                except json.JSONDecodeError:
                    tool_args = {}

                print(
                    f"Calling tool: {tool_name} "
                    f"with args: {tool_args}"
                )

                try:
                    result = execute_tool(
                        tool_name,
                        tool_args
                    )

                except Exception as exc:
                    result = {
                        "error": str(exc)
                    }

                # O output da função precisa voltar
                # associado ao call_id correspondente.
                tool_outputs.append(
                    {
                        "type": "function_call_output",
                        "call_id": item.call_id,
                        "output": json.dumps(
                            result,
                            ensure_ascii=False
                        )
                    }
                )

        # Se não houve function call,
        # temos a resposta final.
        if not tool_outputs:
            return response.output_text

        # Continua a mesma interação enviando
        # o resultado das tools para o modelo.
        response = client.responses.create(
            model=llm_model,
            instructions=SYSTEM_PROMPT,
            previous_response_id=response.id,
            input=tool_outputs,
            tools=tools,
        )



<perfil>
Você é um agente orquestrador especializado em responder perguntas do usuário
utilizando raciocínio, conhecimento próprio e ferramentas externas disponíveis.

Seu objetivo é fornecer respostas corretas, úteis, objetivas e baseadas em dados
confiáveis.

Você faz parte de uma arquitetura agentic, na qual ferramentas podem atuar como
especialistas responsáveis por recuperar ou processar informações específicas.

Você deve decidir autonomamente quando uma ferramenta é necessária.
</perfil>


<tasks>
Suas principais responsabilidades são:

1. Entender a intenção do usuário.

2. Determinar se a pergunta pode ser respondida diretamente ou se necessita
   consultar uma ferramenta.

3. Selecionar a ferramenta mais adequada para cada necessidade.

4. Extrair corretamente os argumentos necessários para chamar a ferramenta.

5. Analisar o resultado retornado pela ferramenta.

6. Caso necessário, realizar novas chamadas de ferramentas.

7. Combinar informações provenientes de diferentes f

In [11]:
def chat_loop():
    print("Type your queries or 'quit' to exit.")
    while True:
        try:
            query = input("\nQuery: ").strip()
            if query.lower() == 'quit':
                break
    
            response = process_query(query)
            print(response)
            print("\n")
            
        except Exception as e:
            print(f"\nError: {str(e)}")

In [ ]:
chat_loop()

Type your queries or 'quit' to exit.



Query:  Qual e o tempo hoje em recife ?


Calling tool: get_current_weather with args: {'location': 'Recife'}
Hoje, em Recife, a temperatura é de aproximadamente 25 °C e há previsão de chuvas leves.





Query:  E na havana qual e o tempo hoje ?


Calling tool: get_current_weather with args: {'location': 'Havana'}
Hoje em Havana, a temperatura é de aproximadamente 29 °C e está ocorrendo uma leve chuva.





Query:  E em viena ?


Calling tool: get_current_weather with args: {'location': 'Viena'}
Em Viena, a temperatura atual é de aproximadamente 17 °C, com céu parcialmente nublado.





Query:  onde a temperatura das que voce falou e 25C?


A temperatura de 25 °C pode ser comum em várias cidades ao redor do mundo, dependendo da época do ano e das condições climáticas. Algumas cidades que frequentemente apresentam temperaturas em torno de 25 °C incluem:

- **Lisboa**, Portugal
- **Barcelona**, Espanha
- **São Paulo**, Brasil (em certas épocas do ano)
- **Cape Town**, África do Sul
- **Brisbane**, Austrália

Se precisar de informações mais específicas sobre uma localização em particular, sinta-se à vontade para perguntar!




In [30]:
# Call the responses endpoint

response = client.chat.completions.create(
        model=llm_model,
        messages=messages,
        tools=tools,
        temperature=0,
    )


In [31]:
print(response)

ChatCompletion(id='chatcmpl-EE0IxB32ufFMVMtmVWEH1Uwbuduaw', choices=[Choice(finish_reason='tool_calls', index=0, logprobs=None, message=ChatCompletionMessage(content=None, refusal=None, role='assistant', annotations=[], audio=None, function_call=None, tool_calls=[ChatCompletionMessageFunctionToolCall(id='call_jLOTVkx4bzdOCoirzjQ09vuZ', function=Function(arguments='{"location":"Recife, Brazil"}', name='get_current_weather'), type='function')]))], created=1787006471, model='gpt-4o-mini-2024-07-18', object='chat.completion', moderation=None, service_tier='default', system_fingerprint='fp_d26e8aed44', usage=CompletionUsage(completion_tokens=18, prompt_tokens=79, total_tokens=97, completion_tokens_details=CompletionTokensDetails(accepted_prediction_tokens=0, audio_tokens=0, reasoning_tokens=0, rejected_prediction_tokens=0), prompt_tokens_details=PromptTokensDetails(audio_tokens=0, cache_write_tokens=None, cached_tokens=0)))


In [32]:
response_message = response.choices[0].message

In [34]:
response_message

ChatCompletionMessage(content=None, refusal=None, role='assistant', annotations=[], audio=None, function_call=None, tool_calls=[ChatCompletionMessageFunctionToolCall(id='call_jLOTVkx4bzdOCoirzjQ09vuZ', function=Function(arguments='{"location":"Recife, Brazil"}', name='get_current_weather'), type='function')])

In [35]:
print(response_message.content)

None


In [36]:
print(response_message.tool_calls)

[ChatCompletionMessageFunctionToolCall(id='call_jLOTVkx4bzdOCoirzjQ09vuZ', function=Function(arguments='{"location":"Recife, Brazil"}', name='get_current_weather'), type='function')]


In [37]:
#for tool_call in response_message.tool_calls:
#    print("Tool Call Name:", tool_call.function.name)
#    print("Tool Call Arguments:", json.loads(tool_call.function.arguments))

print("Function Name:", response_message.tool_calls[0].function.name)
print("Function Arguments:", json.loads(response_message.tool_calls[0].function.arguments))

Function Name: get_current_weather
Function Arguments: {'location': 'Recife, Brazil'}


In [38]:
args = json.loads(response_message.tool_calls[0].function.arguments)
args

{'location': 'Recife, Brazil'}

In [41]:
get_current_weather(**args) # unit="celsius"

'{"location": "Recife, Brazil", "temperature": 77.04, "unit": "fahrenheit", "forecast": ["few clouds"]}'

### LangChain Tools

In [43]:
from langchain.tools import tool
import requests

In [44]:
@tool
def get_weather(city: str, unit: str = "celsius") -> str:
    """
    Obtém a previsão do tempo para a cidade informada.
    Args:
        city: Nome da cidade (ex: 'Recife, PE, Brasil')
        unit: Unidade de temperatura ('fahrenheit', 'celsius', 'kelvin')
    Returns:
        String com resumo da previsão do tempo.
    """

    # Chama API

    # Dummy response
    # resumo = f"[Serviço fictício] A previsão para {city} é: 26°C, céu parcialmente nublado."

    # Real call
    units = {"fahrenheit": "imperial", "celsius": "metric", "kelvin": "standard"}
    if unit not in units:
        raise ValueError("Invalid unit. Must be 'fahrenheit', 'celsius', or 'kelvin'.")

    response = requests.get(f"https://api.openweathermap.org/data/2.5/weather?q={city}&appid={WEATHER_API_KEY}&units={units[unit]}")
    data = response.json()
    weather_info = {
        "location": city,
        "temperature": data['main']['temp'],
        "unit": unit,
        "forecast": [data['weather'][0]['description']],
    }

    resumo = f"A temperatura em {city} agora é {weather_info['temperature']}°{unit[0].upper()} com {', '.join(weather_info['forecast'])}."
    return resumo


@tool
def thinking_tool(reflexion:str):
    """Chame a ferramenta de reflexão para analisar a resposta."""
    return f"[Ferramenta de reflexão] Analisando: {reflexion}"


In [47]:
from langchain_openai import ChatOpenAI

llm = ChatOpenAI(
    model=llm_model,
    temperature=0.0
    )

# Vincula a tool ao modelo
llm_with_tools = llm.bind_tools([thinking_tool, get_weather])

In [50]:
from langchain_core.messages import (
    HumanMessage,
    ToolMessage,
)

user_input = "Qual a previsão do tempo para California hoje em celsius?"

result = llm_with_tools.invoke(
    [HumanMessage(content=user_input)
    ])


# Verifica se houve chamado da tool
if hasattr(result, "tool_calls") and result.tool_calls:

    tool_call = result.tool_calls[0]
    tool_name = tool_call["name"]
    args = tool_call["args"]


    if tool_name != "get_weather":
        raise ValueError(f"Tool inesperada: {tool_name}")

    # Executa a tool
    tool_result = get_weather.invoke(args)

    tool_message = ToolMessage(
        name=tool_name,
        content=str(tool_result),
        tool_call_id=tool_call["id"]
    )


    # Envia de volta ao modelo o resultado da tool, para que ele complete a resposta
    followup = llm_with_tools.invoke([
        HumanMessage(content=user_input),
        result,
        tool_message
    ])

    print(followup.content)

else:
    print(result.content)

    

A previsão do tempo para California, EUA, hoje é de 30.85°C com nuvens fragmentadas.


In [51]:
result.tool_calls[0]

{'name': 'get_weather',
 'args': {'city': 'California, EUA', 'unit': 'celsius'},
 'id': 'call_E8VvGheUKelbDUVXh0JSLfmK',
 'type': 'tool_call'}